In [13]:
import pandas as pd
from pathlib import Path
import altair as alt

In [14]:
df = pd.read_csv('cleaned_listings.csv')

In [15]:
# filter the data to exclude outliers
Q1 = df['minimum_nights'].quantile(0.25)
Q3 = df['minimum_nights'].quantile(0.75)
IQR = Q3 - Q1

threshold = Q3 + 1.5 * IQR
df_filtered2 = df[(df['minimum_nights'] <= threshold) & (df['minimum_nights'] > 15)].copy()
df_filtered1 = df[df['minimum_nights'] <= 15].copy()
# Create bins for availability_365 to show the relationship
categorical_order = ['Low (0-90)', 'Medium (91-180)', 'High (181-270)', 'Very High (271-365)']
df_filtered1['availability_category'] = pd.cut(
    df_filtered1['availability_365'], 
    bins=[0, 90, 180, 270, 365],
    labels=categorical_order
)

In [16]:
color_order = ['null'] + categorical_order

order_map = {cat: i for i, cat in enumerate(color_order)}

df_filtered1 = df_filtered1.copy()
df_filtered1['availability_order'] = df_filtered1['availability_category'].map(order_map)

In [17]:
color_order = ['null'] + categorical_order

chart1 = alt.Chart(df_filtered1).mark_bar(opacity=0.7,  stroke=None   ).encode(
    x=alt.X('minimum_nights:Q', 
            bin=alt.Bin(maxbins=20),
            title='Minimum Nights Required'),
    y=alt.Y('count()', title='Number of Listings'),
    color=alt.Color('availability_category:N',
                    title='Availability (365 days)',
                    scale=alt.Scale(scheme='viridis', domain=color_order), sort=color_order),
    order=alt.Order('availability_order:Q'),
    tooltip=[
        alt.Tooltip('minimum_nights:Q', bin=True, title='Minimum Nights'),
        alt.Tooltip('count()', title='Count'),
        alt.Tooltip('availability_category:N', title='Availability Level')
    ]
).properties(
    width=700,
    height=400,
    title='Distribution of Minimum Nights by Availability Level'
).interactive()

c1_json = chart1.to_json()

with open('website/t4-barplot1_spec.json', 'w') as f:
    f.write(c1_json)

In [18]:
df_filtered2['availability_category'] = pd.cut(
    df_filtered2['availability_365'], 
    bins=[0, 90, 180, 270, 365],
    labels=categorical_order
)

df_filtered2 = df_filtered2.copy()
df_filtered2['availability_order'] = df_filtered2['availability_category'].map(order_map)

chart2 = alt.Chart(df_filtered2).mark_bar(opacity=0.7,  stroke=None   ).encode(
    x=alt.X('minimum_nights:Q', 
            bin=alt.Bin(maxbins=20),
            title='Minimum Nights Required'),
    y=alt.Y('count()', title='Number of Listings'),
    color=alt.Color('availability_category:N',
                    title='Availability (365 days)',
                    scale=alt.Scale(scheme='viridis', domain=color_order), sort=color_order),
    order=alt.Order('availability_order:Q'),
    tooltip=[
        alt.Tooltip('minimum_nights:Q', bin=True, title='Minimum Nights'),
        alt.Tooltip('count()', title='Count'),
        alt.Tooltip('availability_category:N', title='Availability Level')
    ]
).properties(
    width=700,
    height=400,
    title='Distribution of Minimum Nights by Availability Level'
).interactive()

c2_json = chart2.to_json()

with open('website/t4-barplot2_spec.json', 'w') as f:
    f.write(c2_json)